In [1]:
import cv2
import numpy as np
import mediapipe as mp
from collections import deque, Counter

In [2]:
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    model_complexity=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.5
)

In [ ]:
WIDTH = 640
HEIGHT = 480

cap = cv2.VideoCapture(0)

cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    WIDTH
)

cap.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    HEIGHT
)

In [ ]:
import joblib

raw_scaler = joblib.load(
    "../models/raw_scaler.pkl"
)
wrist_centered_linear_svm = joblib.load(
    "../models/raw_linear_svm.pkl"
)

current_model = wrist_centered_linear_svm
current_model_name = "Linear SVM"

In [ ]:
# ---------------------------------
# Prediction history
# ---------------------------------

prediction_history = deque(maxlen=10)


# ---------------------------------
# Sentence
# ---------------------------------

sentence = ""


# ---------------------------------
# Current gesture
# ---------------------------------
# This is the gesture currently
# being held in front of the camera.

current_gesture = None

In [ ]:
def extract_landmarks(hand_landmarks):

    landmarks = []

    for landmark in hand_landmarks.landmark:

        landmarks.extend([
            landmark.x,
            landmark.y,
            landmark.z
        ])

    X = np.array(landmarks).reshape(1, -1)

    return X

In [ ]:
def predict_gesture(X, scaler, model, class_names):

    # Scale landmarks
    X_scaled = scaler.transform(X)

    # Predict class
    prediction = model.predict(X_scaled)

    predicted_class = prediction[0]

    # Convert class number to letter
    label = class_names[predicted_class]

    # Probability
    probabilities = model.predict_proba(X_scaled)

    confidence = np.max(probabilities)

    return label, confidence

In [ ]:
def get_stable_prediction(prediction_history):

    if len(prediction_history) == 0:
        return "No hand"

    stable_label = Counter(
        prediction_history
    ).most_common(1)[0][0]

    return stable_label

In [ ]:
def add_gesture_to_sentence(gesture,sentence):

    # ---------------------------------
    # Nothing to add
    # ---------------------------------

    if gesture is None:
        return sentence


    # ---------------------------------
    # SPACE
    # ---------------------------------

    if gesture == "space":

        sentence += " "


    # ---------------------------------
    # DELETE
    # ---------------------------------

    elif gesture == "del":

        sentence = sentence[:-1]


    # ---------------------------------
    # Normal letter
    # ---------------------------------

    else:

        sentence += gesture


    return sentence

In [ ]:
def draw_text(
    display_frame,
    label,
    confidence,
    stable_label,
    sentence,
    current_model_name
):

    # =====================================
    # Sentence — TOP
    # =====================================

    cv2.rectangle(
        display_frame,
        (10, 10),
        (630, 70),
        (30, 30, 30),
        -1
    )

    cv2.putText(
        display_frame,
        f"Text: {sentence}",
        (20, 52),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (255, 255, 255),
        2
    )


    # =====================================
    # Prediction
    # =====================================

    cv2.putText(
        display_frame,
        f"Prediction: {label}",
        (20, 100),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (0, 255, 0),
        2
    )


    # =====================================
    # Confidence
    # =====================================

    cv2.putText(
        display_frame,
        f"Confidence: {confidence * 100:.2f}%",
        (20, 130),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )


    # =====================================
    # Current Model
    # =====================================

    cv2.putText(
        display_frame,
        f"Model: {current_model_name}",
        (20, 160),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 0),
        2
    )


    # =====================================
    # Controls
    # =====================================

    cv2.putText(
        display_frame,
        "C: Clear | Q: Quit",
        (10, 460),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 0, 255),
        1
    )

In [ ]:
class_names = [
    "A", "B", "C", "D", "E", "F", "G", "H", "I", "J",
    "K", "L", "M", "N", "O", "P", "Q", "R", "S", "T",
    "U", "V", "W", "X", "Y", "Z", "del", "space"
]

In [ ]:
while True:

    # =====================================
    # 1. READ FRAME
    # =====================================

    success, frame = cap.read()

    if not success:

        print("Error: Can't receive frame")
        break


    # =====================================
    # 2. DEFAULT VALUES
    # =====================================

    label = "No hand"
    confidence = 0.0
    stable_label = "No hand"


    # =====================================
    # 3. MEDIAPIPE PROCESSING
    # =====================================

    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    results = hands.process(rgb_frame)


    # =====================================
    # 4. HAND DETECTED
    # =====================================

    if results.multi_hand_landmarks:

        hand_landmarks = results.multi_hand_landmarks[0]


        # ---------------------------------
        # Draw landmarks
        # ---------------------------------

        mp_draw.draw_landmarks(
            frame,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS,

            mp_draw.DrawingSpec(
                color=(0, 255, 0),
                thickness=4,
                circle_radius=5
            ),

            mp_draw.DrawingSpec(
                color=(0, 0, 255),
                thickness=4
            )
        )


        # ---------------------------------
        # Extract landmarks
        # ---------------------------------

        X = extract_landmarks(
            hand_landmarks
        )


        # ---------------------------------
        # Prediction
        # ---------------------------------

        label, confidence = predict_gesture(
            X,
            raw_scaler,
            current_model,
            class_names
        )


        # ---------------------------------
        # Add prediction to history
        # ---------------------------------

        prediction_history.append(label)


        # ---------------------------------
        # Majority vote
        # ---------------------------------

        stable_label = Counter(
            prediction_history
        ).most_common(1)[0][0]


        # ---------------------------------
        # Remember current gesture
        # ---------------------------------

        current_gesture = stable_label


    # =====================================
    # 5. NO HAND
    # =====================================

    else:

        # ---------------------------------
        # If we were holding a gesture,
        # commit it to the sentence
        # ---------------------------------

        if current_gesture is not None:

            sentence = add_gesture_to_sentence(
                current_gesture,
                sentence
            )

            print(
                f"Added: {current_gesture}"
            )

            print(
                f"Sentence: {sentence}"
            )


        # ---------------------------------
        # Reset gesture state
        # ---------------------------------

        current_gesture = None

        prediction_history.clear()


    # =====================================
    # 6. FLIP FOR VISUALIZATION
    # =====================================

    display_frame = cv2.flip(
        frame,
        1
    )


    # =====================================
    # 7. DRAW UI
    # =====================================

    draw_text(
        display_frame,
        label,
        confidence,
        stable_label,
        sentence,
        current_model_name
    )


    # =====================================
    # 8. DISPLAY
    # =====================================

    cv2.imshow(
        "ASL Fingerspelling Translator",
        display_frame
    )


    # =====================================
    # 9. KEYBOARD INPUT
    # =====================================

    key = cv2.waitKey(1) & 0xFF


    # ---------------------------------
    # Quit
    # ---------------------------------

    if key == ord('q'):

        break


    # ---------------------------------
    # Clear sentence
    # ---------------------------------

    elif key == ord('c'):

        sentence = ""

        current_gesture = None

        prediction_history.clear()

c:\Users\Shehab Abdo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\Shehab Abdo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\Shehab Abdo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.w

Added: H
Sentence: H


c:\Users\Shehab Abdo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\Shehab Abdo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\Shehab Abdo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.w

Added: E
Sentence: HE


c:\Users\Shehab Abdo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\Shehab Abdo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\Shehab Abdo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.w

...

Added: G
Sentence: HELG


c:\Users\Shehab Abdo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\Shehab Abdo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\Shehab Abdo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.w

Added: del
Sentence: HEL


c:\Users\Shehab Abdo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\Shehab Abdo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\Shehab Abdo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.w

Added: J
Sentence: HELJ
